# Phenotype Data Cleaning Pipeline
**UK Biobank — G12 (Motor Neuron Disease / ALS) Cohort**

---

This notebook cleans and prepares the raw phenotype data (`pheno_G12_500k.csv`) extracted from the UK Biobank for downstream analysis.

### Steps covered:
1. Load libraries and raw data
2. Parse visit dates (instances 0–3)
3. Construct date of birth (`dob`)
4. Create diagnosis-derived variables: `G12_count`, `ALS`, `G00_G99`
5. Quality checks
6. Save cleaned data


## 1. Setup 


In [ ]:
library(tidyverse)
library(dplyr)
library(lubridate)

## 2. Load Raw Data

The raw phenotype file contains ~500k UK Biobank participants with G12-related diagnostic information, visit dates across up to 4 assessment centre instances, Olink proteomics selection flags, and ICD-10 diagnosis records.

In [ ]:
# load pheno_G12_500k_plate.csv
pheno_raw <- read.csv("pheno_G12_500k.csv", row.names = 1)


## 3. Parse Visit Dates (Instances 0–3)

UK Biobank stores visit dates across **4 assessment instances** (baseline + 3 repeat visits), each with up to **7 array fields** (e.g. `visit_date_i0_0` through `visit_date_i0_6`). These array fields represent multiple measurements recorded during the same visit.

For each instance, we:
- Take the **first non-missing** value across all 7 array fields
- Strip the timestamp (format is `2009-10-08T14:44:21`), keeping only the **date** (`YYYY-MM-DD`)
- Drop all original array columns after extraction

In [ ]:
for (i in 0:3) {
  array_cols <- paste0("visit_date_i", i, "_", 0:6)
  new_col    <- paste0("visit_date_i", i)
  
  pheno_raw[[new_col]] <- as.Date(apply(
    pheno_raw[, array_cols, drop = FALSE], 1, function(x) {
      first_val <- x[!is.na(x) & x != ""][1]
      if (is.na(first_val) || is.null(first_val)) return(NA_character_)
      substr(first_val, 1, 10)  # "2009-10-08T14:44:21" -> "2009-10-08"
    }
  ))
  
  pheno_raw[, array_cols] <- NULL
}

In [ ]:
colnames(pheno_raw)

## 4. Construct Date of Birth (`dob`)

UK Biobank does not provide exact dates of birth for privacy reasons. Instead, it provides:
- `yob`: year of birth (integer, e.g. `1947`)
- `mob`: month of birth (full name, e.g. `"January"`)

We combine these into an approximate `dob` by assigning **day 15** as a neutral mid-month placeholder. The result is stored as a `Date` object (`YYYY-MM-DD`) consistent with the visit date columns.

In [ ]:
# 1. Create dob from yob + mob
pheno_raw <- pheno_raw %>%
  mutate(
    dob = as.Date(paste0(yob, "-", mob, "-15"), format = "%Y-%B-%d"),
    age = as.numeric(visit_date_i0 - dob) / 365.25
  )

In [ ]:
# make a histogram of the age
hist(pheno_raw$age)


## 5. Derive Diagnosis Variables

The `diagnoses` column contains a participant's full ICD-10 diagnostic history as a string. We extract three variables:

| Variable | Description |
|---|---|
| `G12_count` | Number of G12 (Spinal muscular atrophy / MND) codes in diagnoses |
| `ALS` | Binary flag: `1` if ICD-10 code `G12.2` (Motor neurone disease) is present |
| `G00_G99` | Binary flag: `1` if any neurological disorder (G00–G99) is present |

### ICD-10 Chapter G: Diseases of the Nervous System
- **G12**: Spinal muscular atrophy and related syndromes
- **G12.2**: Motor neurone disease (ALS / MND)
- **G00–G99**: Covers all neurological conditions (used as a broader neurological comorbidity flag)

In [ ]:

# 2. Count G12 diagnoses and flag ALS (G12.2)
pheno_raw <- pheno_raw %>%
  mutate(
    # Count how many G12 codes appear in diagnoses
    G12_count = stringr::str_count(diagnoses, "G12"),
    
    # ALS = 1 if G12.2 present, 0 otherwise
    ALS = as.integer(
    stringr::str_detect(diagnoses, "G12\\.2") |
    stringr::str_detect(death_reason_major, "G12\\.2")
  ),

    G00_G99 = as.integer(stringr::str_detect(diagnoses, "\\bG\\d{2}"))
  )

  

## 6. Quality Checks

Before saving, we verify the cleaned data looks as expected:
- Inspect column names and a preview of the data
- Check counts of G12 diagnoses and ALS cases
- Verify `G12_date` is populated for participants with a G12 diagnosis
- Confirm all ALS cases have a matching `G12_date` (internal consistency check)

In [ ]:
# drop the diagnosis column
pheno_raw <- pheno_raw %>% select(-diagnoses)
# drop 'olink_counts_0''olink_counts_1''olink_counts_2''olink_counts_3''UKB_PPP_select_i0''UKB_PPP_select_i1''UKB_PPP_select_i2''UKB_PPP_select_i3''visit_date_i0''visit_date_i1''visit_date_i2''visit_date_i3'
pheno_raw <- pheno_raw %>% select(-olink_counts_0, -olink_counts_1, -olink_counts_2, -olink_counts_3, -UKB_PPP_select_i0, -UKB_PPP_select_i1, -UKB_PPP_select_i2, -UKB_PPP_select_i3, )
# drop the 'yob''mob'column
pheno_raw <- pheno_raw %>% select(-yob, -mob)

In [ ]:
colnames(pheno_raw)

In [ ]:
# check G12_date not ""
sum(pheno_raw$G12_date != "")



In [ ]:
table(pheno_raw$G12_count)
table(pheno_raw$ALS)


In [ ]:
# check the G00_G99 column
table(pheno_raw$G00_G99)


In [ ]:
# check if all ALS are 1 with G12_date not ""
sum(pheno_raw$ALS == 1 & pheno_raw$G12_date != "")



## 8. Time Variables: Years to Diagnosis and Censoring

Two time variables are derived relative to the baseline visit (`visit_date_i0`):

| Variable | Description |
|---|---|
| `YrSinceDi` | Years between baseline and ALS diagnosis (G12.2). Negative = baseline before diagnosis (pre-diagnostic), positive = baseline after diagnosis. Only populated for ALS cases with a single G12 code. |
| `YrSinceCen` | Years between baseline and censoring event (always negative). Censoring hierarchy: lost to follow-up > death > study end (2025-09-01). |

In [ ]:
pheno_raw <- pheno_raw %>%
  mutate(
    G12_date   = as.Date(ifelse(G12_date == "", NA, G12_date)),
    lost_date  = as.Date(ifelse(lost_date == "", NA, lost_date)),
    death_date = as.Date(ifelse(death_date == "", NA, death_date)),
    
    # Negative = baseline before diagnosis, positive = baseline after diagnosis
    YrSinceDi = ifelse(
      ALS == 1 & G12_count == 1,
      as.numeric(difftime(visit_date_i0, G12_date, units = "days")) / 365.25,
      NA_real_
    ),
    
    censor_date = case_when(
      !is.na(lost_date)  ~ lost_date,
      !is.na(death_date) ~ death_date,
      TRUE               ~ as.Date("2025-09-01")
    ),
    # Always negative: baseline is always before censoring
    YrSinceCen = as.numeric(difftime(visit_date_i0, censor_date, units = "days")) / 365.25
  ) %>%
  select(-censor_date)

### Distribution of Time Variables

Histograms below show the spread of `YrSinceDi` (ALS cases only) and `YrSinceCen` (all participants).
A concentration of `YrSinceDi` around 0 would suggest many participants were recruited close to their diagnosis date.

In [ ]:
ggplot(pheno_raw %>% filter(!is.na(YrSinceDi)), aes(x = YrSinceDi)) +
  geom_histogram(binwidth = 1, fill = "firebrick", color = "white") +
  geom_vline(xintercept = 0, linetype = "dashed", color = "black") +
  labs(
    title = "Years from Baseline to ALS Diagnosis",
    subtitle = "ALS cases only (G12.2, G12_count = 1)",
    x = "Years Since Diagnosis (negative = pre-diagnostic)",
    y = "Count"
  ) +
  theme_minimal()

In [ ]:
# random select one YrSinceDi, show the visit_date_i0, G12_date, YrSinceDi
pheno_raw %>% filter(!is.na(YrSinceDi)) %>% sample_n(1) %>% select(visit_date_i0, G12_date, YrSinceDi)



In [ ]:
ggplot(pheno_raw, aes(x = YrSinceCen)) +
  geom_histogram(binwidth = 1, fill = "steelblue", color = "white") +
  geom_vline(xintercept = 0, linetype = "dashed", color = "black") +
  labs(
    title = "Years from Baseline to Censoring",
    subtitle = "All participants — lost / death / study end (2025-09-01)",
    x = "Years Since Censoring (always negative)",
    y = "Count"
  ) +
  theme_minimal()


## 7. Save Cleaned Data

The cleaned phenotype file is saved as a CSV for use in downstream analysis pipelines (e.g. proteomics matching, survival analysis).

In [ ]:
write.csv(pheno_raw, here::here("data", "analysis_data", "ukb", "visit_info", "cleaned_pheno.csv"))